# 🌡️ 01a — Pipeline météo

Construit `dim_meteo.parquet` (dept × annee_mois). Lancer `00_config_commun.ipynb` avant.

Source : Météo France, données climatologiques mensuelles (MENSQ).
https://www.data.gouv.fr/datasets/donnees-climatologiques-de-base-mensuelles
Fichiers dans `data/raw/MeteoFrance/MENSQ_XX_{previous-1950-2024,latest-2025-2026}.csv.gz`,
un par département (`XX`), Corse regroupée sous le code `20` à séparer en 2A/2B.

On utilisait SYNOP avant (42 stations, 41 départements couverts). MENSQ est
plus dense et déjà agrégé au mois par station, donc plus besoin de ré-agréger
des relevés horaires, et on couvre les 96 départements une fois la Corse
séparée en 2A/2B.

Correspondance des colonnes (justification détaillée du choix de variables
juste avant `build_dim_meteo` ci-dessous) :

**Température** — détermine la floraison (allergie) ; l'air froid déclenche des crises (asthme)

| Champ | Description | Justification |
|---|---|---|
| `TM` | Température moyenne mensuelle | Signal thermique global du mois |
| `TX` | Moyenne des températures maximales | Une chaleur élevée accélère la libération du pollen |
| `TN` | Moyenne des températures minimales | Un gel nocturne retarde la floraison ; le froid déclenche aussi des bronchospasmes (asthme) |
| `TAMPLIM` | Amplitude thermique diurne moyenne | Une forte amplitude stresse les plantes → grains de pollen plus fragiles ; irrite aussi les voies respiratoires |
| `NBJGELEE` | Nombre de jours de gelée | Le gel tue ou retarde la floraison |
| `NBJTX0` | Nb jours où TX ≤ 0°C | L'air froid est un déclencheur classique de crise d'asthme |
| `NBJTX25`, `NBJTX30` | Nb jours de forte chaleur (≥25°C/≥30°C) | Les pics de chaleur coïncident souvent avec les pics de pollen et aggravent la pollution à l'ozone (asthme) |

**Précipitations** — lessivage de l'air vs. stimulation de la croissance végétale

| Champ | Description | Justification |
|---|---|---|
| `RR` | Cumul mensuel des précipitations | La pluie lessive le pollen à court terme, mais stimule la croissance végétale (hausse différée) |
| `NBJRR1` | Nb jours avec RR ≥ 1 mm | Reflète mieux la fréquence du lessivage que le cumul seul |
| `NBJRR5` | Nb jours avec RR ≥ 5 mm | Épisodes pluvieux plus marqués |

**Humidité** — éclatement des grains de pollen, spores fongiques

| Champ | Description | Justification |
|---|---|---|
| `UMM` | Humidité relative moyenne | Une forte humidité peut faire éclater les grains de pollen en particules plus fines et plus allergisantes (mécanisme de « l'asthme d'orage »), et favorise les spores fongiques |

**Vent** — dispersion du pollen

| Champ | Description | Justification |
|---|---|---|
| `FFM` | Vitesse moyenne du vent | Moteur principal de dispersion du pollen |
| `FXIAB` | Rafale maximale du mois | Le vent fort disperse le pollen plus loin |
| `NBJFF10` | Nb jours où le vent ≥ 10 m/s | Fréquence des épisodes venteux |

**Ensoleillement / rayonnement** — favorise la libération du pollen

| Champ | Description | Justification |
|---|---|---|
| `INST` | Cumul de la durée d'ensoleillement (**minutes**), convertie en heures dans `build_dim_meteo` | Le beau temps favorise la libération active du pollen |
| `GLOT` | Cumul du rayonnement global | Lié à la photosynthèse/floraison ; indicateur indirect de pollution photochimique |

**Événements météo particuliers** — situations à risque élevé communes à l'allergie et à l'asthme

| Champ | Description | Justification |
|---|---|---|
| `NBJORAG` | Nombre de jours d'orage | « L'asthme d'orage » (thunderstorm asthma) est documenté dans la littérature : l'orage fait éclater les grains de pollen, provoquant un pic d'urgences allergie/asthme dans les heures qui suivent — variable jugée particulièrement importante |
| `NBJBROU` | Nombre de jours de brouillard | Le brouillard piège polluants/allergènes près du sol, aggravant allergie et asthme |
| `PMERM` | Pression moyenne au niveau de la mer | Un système de haute pression stable favorise l'accumulation de polluants près du sol (déclencheur connu de crises d'asthme) ; les variations brutales de pression irritent aussi les voies respiratoires |

**Variable secondaire** (pertinence à confirmer empiriquement)

| Champ | Description | Justification |
|---|---|---|
| `ETP` | Évapotranspiration potentielle | Indicateur indirect du stress hydrique des plantes, lié surtout à la production de pollen (allergie) |

### Variables exclues (et pourquoi)

- **Champs `*DAT`** (`RRABDAT`, `TXDAT`, `TNDAT`, `FXIDAT`, …) : ne sont que
  la date du jour où l'extrême a été observé — sans pouvoir explicatif à
  l'échelle mensuelle.
- **Champs de complétude `NB*`** (`NBRR`, `NBTX`, `NBTN`, `NBPMERM`, `NBUM`,
  `NBFFM`, …) : comptent le nombre de jours avec observation valide dans le
  mois — un indicateur de qualité des données, pas un signal climatique.
- **Champs neige** (`HNEIGEFTOT`, `NEIGETOTM`, `NBJNEIG`, `NBJHNEIGEF*`,
  `NBJSOLNG`, …) : hors saison pollinique dans la plupart des cas, sauf étude
  spécifique en zone de montagne.
- **Rayonnement détaillé** (`DIFT` diffus, `DIRT` direct) : redondant avec
  `GLOT` (total), risque de colinéarité.
- **Températures redondantes/estimées** (`TX_ME`, `TN_ME`, `TMM`, `TMMIN`,
  `TMMAX`, `TXMIN`, `TNMAX`) : très corrélées à `TM`/`TX`/`TN`, exclues pour
  limiter la colinéarité sauf besoin d'un extrême précis.
- **Variantes de vent** (`FXI3SAB`, `FXYAB` et dérivés) : mesurent le même
  phénomène que `FXIAB`/`FFM` sur d'autres pas de temps — un seul indicateur
  de rafale et un seul de vent moyen suffisent.
- **Extrêmes d'humidité** (`UNAB`, `UXAB`) : redondants avec `UMM`, à
  réintroduire seulement si un besoin spécifique d'humidité extrême apparaît.
- **Grêle** (`NBJGREL`) : événement rare, lien direct avec l'allergie faible,
  échantillon probablement trop petit.
- **Direction du vent** (`DXIAB`, `DXYAB`, `DXI3SAB`) : utile seulement en
  croisement avec les sources de pollen connues (ex. forêts/cultures en
  amont) — hors périmètre d'un modèle national/régional standard.

In [39]:
# Préambule : on se place dans le répertoire racine du projet et on ajoute le répertoire courant au PYTHONPATH pour pouvoir importer src/config.py

# pour recharger automatiquement les modules modifiés （src config surtout） sans redémarrer le kernel
%load_ext autoreload 
%autoreload 2

import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS
from src.validation import valider_dim_table

print(
    f"Config chargée depuis src/config.py : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"RAW_DIR    = {RAW_DIR}")
print(f"TABLES_DIR = {TABLES_DIR}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Config chargée depuis src/config.py : 96 départements | 2020–2025
RAW_DIR    = /Users/siranh/Documents/Data Scientest/projet_liora/data/raw
TABLES_DIR = /Users/siranh/Documents/Data Scientest/projet_liora/data/processed


In [40]:
# Vérifier la structure d'un fichier MENSQ (1 fichier = 1 département)
df_check = pd.read_csv(
    "data/raw/MeteoFrance/MENSQ_01_previous-1950-2024.csv.gz",
    sep=";", compression="gzip", nrows=5)
print("=== MENSQ (dept 01) ===")
print(df_check.columns.tolist())
display(df_check[["NUM_POSTE", "NOM_USUEL", "LAT", "LON", "AAAAMM",
                   "TM", "TX", "TN", "UMM", "FFM", "RR", "QRR"]].head())
# QRR = 1, ok données valides

=== MENSQ (dept 01) ===
['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'AAAAMM', 'RR', 'QRR', 'NBRR', 'RR_ME', 'RRAB', 'QRRAB', 'RRABDAT', 'NBJRR1', 'NBJRR5', 'NBJRR10', 'NBJRR30', 'NBJRR50', 'NBJRR100', 'PMERM', 'QPMERM', 'NBPMERM', 'PMERMINAB', 'QPMERMINAB', 'PMERMINABDAT', 'TX', 'QTX', 'NBTX', 'TX_ME', 'TXAB', 'QTXAB', 'TXDAT', 'TXMIN', 'QTXMIN', 'TXMINDAT', 'NBJTX0', 'NBJTX25', 'NBJTX30', 'NBJTX35', 'NBJTXI20', 'NBJTXI27', 'NBJTXS32', 'TN', 'QTN', 'NBTN', 'TN_ME', 'TNAB', 'QTNAB', 'TNDAT', 'TNMAX', 'QTNMAX', 'TNMAXDAT', 'NBJTN5', 'NBJTN10', 'NBJTNI10', 'NBJTNI15', 'NBJTNI20', 'NBJTNS20', 'NBJTNS25', 'NBJGELEE', 'TAMPLIM', 'QTAMPLIM', 'TAMPLIAB', 'QTAMPLIAB', 'TAMPLIABDAT', 'NBTAMPLI', 'TM', 'QTM', 'NBTM', 'TMM', 'QTMM', 'NBTMM', 'NBJTMS24', 'TMMIN', 'QTMMIN', 'TMMINDAT', 'TMMAX', 'QTMMAX', 'TMMAXDAT', 'UNAB', 'QUNAB', 'UNABDAT', 'NBUN', 'UXAB', 'QUXAB', 'UXABDAT', 'NBUX', 'UMM', 'QUMM', 'NBUM', 'TSVM', 'QTSVM', 'NBTSVM', 'ETP', 'QETP', 'FXIAB', 'QFXIAB', 'DXIAB', 'QDXIAB', 'FXIDA

,NUM_POSTE,NOM_USUEL,LAT,LON,AAAAMM,TM,TX,TN,UMM,FFM,RR,QRR
0,1010001,ANGLEFORT,45.913667,5.809833,195001,NaN,NaN,NaN,NaN,NaN,49.5,1
1,1010001,ANGLEFORT,45.913667,5.809833,195002,NaN,NaN,NaN,NaN,NaN,237.6,1
2,1010001,ANGLEFORT,45.913667,5.809833,195003,NaN,NaN,NaN,NaN,NaN,23.8,1
3,1010001,ANGLEFORT,45.913667,5.809833,195005,NaN,NaN,NaN,NaN,NaN,83.6,1
4,1010001,ANGLEFORT,45.913667,5.809833,195006,NaN,NaN,NaN,NaN,NaN,75.3,1


In [41]:
import reverse_geocoder as rg
from src.config import DEPT_NOM_TO_CODE


def split_corse_2a_2b(df_20: pd.DataFrame) -> dict:
    """
    Le fichier MENSQ_20_* regroupe toutes les stations de Corse sous un seul code département "20", sans distinction 2A (Corse-du-Sud) / 2B (Haute-Corse).

    On identifie le bon département pour chaque station via un reverse geocoding sur sa latitude/longitude.

    Retourne : dict { NUM_POSTE (int) → "2A" ou "2B" }
    """
    stations = df_20[["NUM_POSTE", "LAT", "LON"]].drop_duplicates().copy()

    coords = list(zip(stations["LAT"], stations["LON"]))
    resultats = rg.search(coords)
    dept_nom = [r["admin2"] for r in resultats]

    # Nom département → code INSEE (cf. src/config.py) ; seuls 2A/2B nous
    # intéressent ici, le reste renverra None et sera filtré par le caller.
    stations["dept"] = [DEPT_NOM_TO_CODE.get(n) for n in dept_nom]

    non_resolues = stations["dept"].isna().sum()
    if non_resolues:
        print(f"  ⚠️  {non_resolues} stations corses non résolues (ignorées)")
    else:
        print(f"  ✅ {len(stations)} stations corses réparties en 2A/2B")

    return dict(zip(stations["NUM_POSTE"], stations["dept"]))

La cible (urgences) est mensuelle et on veut capter la saisonnalité fine
(allergie/asthme au printemps / en été). MENSQ étant déjà mensuel par
station, il reste juste à moyenner les stations d'un même département sur
le même mois.

In [42]:
# Contrôle qualité : distribution des codes qualité (Qxxx) des variables retenues.
# Chaque paramètre MENSQ a un code associé qui qualifie la validation de la donnée :
#   9 = filtrée  (a passé les contrôles de premier niveau, pas encore validée)
#   1 = validée  (contrôle automatique ou climatologue)
#   0 = protégée (validée définitivement par le climatologue)
#   2 = douteuse (mise en doute par contrôle automatique, en cours de vérification)
# Diagnostic uniquement : on affiche la distribution, on ne filtre rien pour l'instant.

Q_COLS = ["QTM", "QTX", "QTN", "QTAMPLIM", "QUMM", "QFFM",     "QFXIAB","QRR", "QPMERM", "QINST", "QGLOT", "QETP"]
meteo_dir = RAW_DIR / "MeteoFrance"
periodes = ["previous-1950-2024", "latest-2025-2026"]
cols = Q_COLS + ["AAAAMM"]

frames = []
for code in [f"{i:02d}" for i in range(1, 96)]:
    for periode in periodes:
        fpath = meteo_dir / f"MENSQ_{code}_{periode}.csv.gz"
        if fpath.exists():
            df_p = pd.read_csv(
                fpath, sep=";", compression="gzip",
                usecols=lambda c: c in cols, low_memory=False
            )
            frames.append(df_p)

df_q = pd.concat(frames)
df_q["annee"] = df_q["AAAAMM"] // 100
df_q = df_q[(df_q["annee"] >= ANNEE_DEBUT) & (df_q["annee"] <= ANNEE_FIN)]

print(f"Contrôle qualité sur {len(df_q):,} lignes station×mois "
    f"({ANNEE_DEBUT}-{ANNEE_FIN}, tous départements)\n")

lignes = []
for col in Q_COLS:
    dist = df_q[col].value_counts(
        dropna=True, normalize=True).mul(100).round(2)
    lignes.append({
        "champ": col,
        "validée (1) %": dist.get(1, 0.0),
        "protégée (0) %": dist.get(0, 0.0),
        "filtrée (9) %": dist.get(9, 0.0),
        "douteuse (2) %": dist.get(2, 0.0),
        "sans mesure %": round(df_q[col].isna().mean() * 100, 2),
    })

display(pd.DataFrame(lignes).set_index("champ").sort_index())

Contrôle qualité sur 158,713 lignes station×mois (2020-2025, tous départements)



,validée (1) %,protégée (0) %,filtrée (9) %,douteuse (2) %,sans mesure %
champ,,,,,
QETP,100.00,0.00,0.00,0.00,14.81
QFFM,99.88,0.00,0.12,0.00,66.72
QFXIAB,99.92,0.05,0.02,0.00,66.76
QGLOT,0.42,0.00,99.50,0.08,90.98
QINST,0.14,0.00,99.86,0.00,92.38
QPMERM,99.05,0.00,0.95,0.00,92.24
QRR,99.30,0.01,0.68,0.00,3.05
QTAMPLIM,99.37,0.00,0.62,0.01,14.26
QTM,99.33,0.00,0.67,0.00,14.26


In [44]:
def build_dim_meteo() -> pd.DataFrame:
    """
    Construit la table dim_meteo à partir des fichiers mensuels MENSQ de
    Météo France (un fichier par département, déjà agrégé au mois par station).

    Etapes:
    ───────────
    1. Charger, pour chaque département 01-95 (dont "20" pour la Corse),
       les fichiers "previous-1950-2024" et "latest-2025-2026"
    2. Filtrer sur la période ANNEE_DEBUT-ANNEE_FIN
    3. Cas particulier "20" : séparer les stations en 2A/2B (reverse geocoding,
       cf. split_corse_2a_2b ci-dessus)
    4. Renommer les champs MENSQ
    5. Agréger les stations d'un même département/mois → moyenne
    6. Sauvegarder en parquet

    VARIABLES PRODUITES (cf. cellule de sélection des variables ci-dessus
    pour la justification de chacune) :
    - dept : code département (str)
    - annee_mois : "YYYY-MM" (str)
    - temp_moy : température moyenne (°C)
    - temp_max : température maximale (°C)
    - temp_min : température minimale (°C)
    - amplitude_thermique_moy : amplitude thermique diurne moyenne (°C)
    - nb_jours_gelee : nombre de jours de gelée (TN < 0°C)
    - nb_jours_sans_degel : nombre de jours sans dégel (TX ≤ 0°C)
    - nb_jours_chaud25 : nombre de jours où TX ≥ 25°C
    - nb_jours_chaud30 : nombre de jours où TX ≥ 30°C
    - humidite_moy : humidité moyenne (%)
    - vent_moy : vitesse moyenne du vent (m/s)
    - vent_rafale_max : rafale maximale du mois (m/s)
    - nb_jours_vent_fort : nombre de jours où le vent moyen ≥ 10 m/s
    - precip_total : précipitations totales (mm)
    - nb_jours_pluie1mm : nombre de jours où RR ≥ 1 mm
    - nb_jours_pluie5mm : nombre de jours où RR ≥ 5 mm
    - ensoleillement_total : cumul mensuel d'ensoleillement (h) — champ source
      INST en minutes, converti en heures (cf. ÉTAPE 4bis)
    - rayonnement_total : cumul mensuel du rayonnement global (J/cm²)
    - nb_jours_orage : nombre de jours d'orage
    - nb_jours_brouillard : nombre de jours de brouillard
    - pression_moy : pression moyenne au niveau de la mer (hPa)
    - etp_total : évapotranspiration potentielle cumulée (mm)

    CLÉ PRIMAIRE : dept × annee_mois
    """
    meteo_dir = RAW_DIR / "MeteoFrance"
    periodes = ["previous-1950-2024", "latest-2025-2026"]

    cols_utiles = ["NUM_POSTE", "LAT", "LON", "AAAAMM",
                   "TM", "TX", "TN", "TAMPLIM", "NBJGELEE", "NBJTX0", "NBJTX25", "NBJTX30",
                   "UMM", "FFM", "FXIAB", "NBJFF10",
                   "RR", "NBJRR1", "NBJRR5",
                   "INST", "GLOT",
                   "NBJORAG", "NBJBROU", "PMERM", "ETP"]

    RENAME_MAP = {
        "TM": "temp_moy",
        "TX": "temp_max",
        "TN": "temp_min",
        "TAMPLIM": "amplitude_thermique_moy",
        "NBJGELEE": "nb_jours_gelee",
        "NBJTX0": "nb_jours_sans_degel",
        "NBJTX25": "nb_jours_chaud25",
        "NBJTX30": "nb_jours_chaud30",
        "UMM": "humidite_moy",
        "FFM": "vent_moy",
        "FXIAB": "vent_rafale_max",
        "NBJFF10": "nb_jours_vent_fort",
        "RR": "precip_total",
        "NBJRR1": "nb_jours_pluie1mm",
        "NBJRR5": "nb_jours_pluie5mm",
        "INST": "ensoleillement_total",
        "GLOT": "rayonnement_total",
        "NBJORAG": "nb_jours_orage",
        "NBJBROU": "nb_jours_brouillard",
        "PMERM": "pression_moy",
        "ETP": "etp_total",
    }

    # ── ÉTAPE 1 : Charger chaque département (01-95, incl. "20" pour Corse) ──
    dfs = []
    df_20 = None  # traité à part le temps de la scission 2A/2B

    for code in [f"{i:02d}" for i in range(1, 96)]:
        frames = []
        for periode in periodes:
            fpath = meteo_dir / f"MENSQ_{code}_{periode}.csv.gz"
            if fpath.exists():
                df_periode = pd.read_csv(
                    fpath, sep=";", compression="gzip",
                    usecols=lambda c: c in cols_utiles, low_memory=False
                )
                frames.append(df_periode)

        if len(frames) == 0:
            print(f"  ⚠️  Aucun fichier pour le département {code}")
            continue

        df_code = pd.concat(frames, ignore_index=True)
        df_code["annee"] = df_code["AAAAMM"] // 100
        df_code = df_code[(df_code["annee"] >= ANNEE_DEBUT) & (df_code["annee"] <= ANNEE_FIN)]

        if code == "20":
            df_20 = df_code
        else:
            df_code = df_code.copy()
            df_code["dept"] = code
            dfs.append(df_code)

    # ── ÉTAPE 2 : Cas particulier de la Corse (scission 2A/2B) ───────────────
    if df_20 is not None and not df_20.empty:
        station_to_dept_corse = split_corse_2a_2b(df_20)
        df_20 = df_20.copy()
        df_20["dept"] = df_20["NUM_POSTE"].map(station_to_dept_corse)
        df_20 = df_20.dropna(subset=["dept"])
        dfs.append(df_20)

    if not dfs:
        print("❌ Aucun fichier MENSQ trouvé dans data/raw/MeteoFrance/")
        return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True)
    print(f"  Total brut {ANNEE_DEBUT}-{ANNEE_FIN} (tous départements) : {len(df):,} lignes")

    # ── ÉTAPE 3 : annee_mois = clé de jointure avec les autres tables ────────
    aaaamm_str = df["AAAAMM"].astype(str)
    df["annee_mois"] = aaaamm_str.str[:4] + "-" + aaaamm_str.str[4:6]

    # ── ÉTAPE 4 : Conversion numérique + renommage vers le schéma existant ──
    for col in RENAME_MAP:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.rename(columns=RENAME_MAP)

    # ── ÉTAPE 4bis : correction d'unité — INST (Météo France) ────────────────
    # Doc officielle MENSQ : "cumul mensuel des durées totales d'insolation
    # quotidiennes", en MINUTES (pas en heures, contrairement à ce qu'on avait
    # supposé au départ). Vérifié empiriquement sur dept 13/juillet 2025 :
    # INST=23140 -> 23140/60 ≈ 386h, cohérent avec un mois très ensoleillé en
    # Méditerranée. On convertit ici pour que la colonne soit bien en heures.
    df["ensoleillement_total"] = df["ensoleillement_total"] / 60

    # ── ÉTAPE 5 : Agrégation des stations d'un même dept/mois → moyenne ─────
    # (chaque valeur MENSQ est déjà une moyenne mensuelle PAR STATION ;
    # ici on moyenne simplement entre les stations d'un même département)
    cols_sortie = list(RENAME_MAP.values())
    df_agg = (
        df.groupby(["dept", "annee_mois"])[cols_sortie]
        .mean()
        .round(2)
        .reset_index()
    )

    print(f"  ✅ dim_meteo : {df_agg.shape[0]:,} lignes × {df_agg.shape[1]} colonnes")
    print(f"  Période      : {df_agg['annee_mois'].min()} → {df_agg['annee_mois'].max()}")
    print(f"  Départements : {df_agg['dept'].nunique()} / {len(DEPTS)} couverts")
    return df_agg

In [45]:
# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════

print("Construction de dim_meteo...")
dim_meteo = build_dim_meteo()

if not dim_meteo.empty:
    valider_dim_table(dim_meteo, "dim_meteo")
    dim_meteo.to_parquet(TABLES_DIR / "dim_meteo.parquet", index=False)
    print(f"\n Sauvegardé → data/processed/dim_meteo.parquet")
    display(dim_meteo.head(10))

Construction de dim_meteo...
  ✅ 81 stations corses réparties en 2A/2B
  Total brut 2020-2025 (tous départements) : 158,713 lignes
  ✅ dim_meteo : 6,912 lignes × 23 colonnes
  Période      : 2020-01 → 2025-12
  Départements : 96 / 96 couverts
── Validation de dim_meteo ──
  ✅ Tous les codes dept sont valides (96 départements)
  ✅ Aucun doublon sur la clé ['dept', 'annee_mois']
  Taux de valeurs manquantes :
    humidite_moy                     3.1%
    vent_moy                         3.1%
    vent_rafale_max                  3.1%
    nb_jours_vent_fort               3.1%
    ensoleillement_total             4.5%
    rayonnement_total                4.6%
    nb_jours_orage                  17.4%
    nb_jours_brouillard              5.1%
    pression_moy                     6.2%
  ✅ OK — prêt pour la fusion (dim_meteo)


 Sauvegardé → data/processed/dim_meteo.parquet


,dept,annee_mois,temp_moy,temp_max,temp_min,amplitude_thermique_moy,nb_jours_gelee,nb_jours_sans_degel,nb_jours_chaud25,nb_jours_chaud30,...,nb_jours_vent_fort,precip_total,nb_jours_pluie1mm,nb_jours_pluie5mm,ensoleillement_total,rayonnement_total,nb_jours_orage,nb_jours_brouillard,pression_moy,etp_total
0,01,2020-01,4.12,8.25,-0.02,8.28,18.10,0.45,0.00,0.00,...,9.00,60.11,8.22,4.25,104.77,14034.00,1.00,5.54,1027.8,19.20
1,01,2020-02,6.92,11.62,2.22,9.40,9.55,0.05,0.00,0.00,...,15.50,106.67,13.14,7.17,124.61,21693.00,2.00,2.67,1023.9,36.51
2,01,2020-03,7.54,13.10,1.98,11.12,9.60,0.00,0.00,0.00,...,13.33,94.82,7.56,5.06,191.93,40627.00,1.12,1.25,1018.0,58.03
3,01,2020-04,13.10,20.39,5.74,14.47,3.75,0.00,2.75,0.00,...,6.67,56.62,4.80,3.29,248.78,55898.75,1.55,1.25,1016.3,95.82
4,01,2020-05,15.24,21.60,8.88,12.72,0.00,0.00,9.35,0.50,...,15.33,96.62,7.89,5.81,253.15,64823.50,1.75,2.88,1018.1,120.36
5,01,2020-06,17.62,23.27,11.98,11.28,0.00,0.00,11.65,2.90,...,9.17,107.39,12.50,6.69,188.89,57609.00,3.41,2.57,1014.3,115.73
6,01,2020-07,21.26,28.72,13.84,14.88,0.00,0.00,24.35,12.80,...,12.50,23.59,4.56,1.11,295.59,73838.50,3.33,1.67,1017.1,157.70
7,01,2020-08,21.62,28.79,14.46,14.31,0.00,0.00,23.25,14.00,...,12.00,59.68,7.53,3.22,234.60,57927.75,2.47,2.50,1014.5,131.78
8,01,2020-09,17.68,24.14,11.23,12.90,0.10,0.00,15.45,6.75,...,8.67,93.63,9.72,6.19,209.26,44776.75,2.76,2.36,1017.2,87.07
9,01,2020-10,10.74,14.65,6.82,7.83,1.25,0.00,0.00,0.00,...,11.83,220.04,15.61,9.03,78.37,20353.00,1.00,4.50,1015.9,38.76


In [46]:
print(dim_meteo.shape)
print(type(dim_meteo))
print(dim_meteo.columns.tolist() if hasattr(
    dim_meteo, "columns") else "not a dataframe")

(6912, 23)
<class 'pandas.core.frame.DataFrame'>
['dept', 'annee_mois', 'temp_moy', 'temp_max', 'temp_min', 'amplitude_thermique_moy', 'nb_jours_gelee', 'nb_jours_sans_degel', 'nb_jours_chaud25', 'nb_jours_chaud30', 'humidite_moy', 'vent_moy', 'vent_rafale_max', 'nb_jours_vent_fort', 'precip_total', 'nb_jours_pluie1mm', 'nb_jours_pluie5mm', 'ensoleillement_total', 'rayonnement_total', 'nb_jours_orage', 'nb_jours_brouillard', 'pression_moy', 'etp_total']


In [47]:
# Vérification de couverture : plus aucun département manquant ?
depts_manquants = sorted(set(DEPTS) - set(dim_meteo["dept"].unique()))
print(f"Départements sans aucune donnée météo : {depts_manquants or 'aucun ✅'}")

doublons = dim_meteo.duplicated(subset=["dept", "annee_mois"]).sum()
print(f"Doublons (dept, annee_mois) : {doublons}")

print("\nTaux de valeurs manquantes par colonne :")
print((dim_meteo.isna().mean() * 100).round(1))

Départements sans aucune donnée météo : aucun ✅
Doublons (dept, annee_mois) : 0

Taux de valeurs manquantes par colonne :
dept                        0.0
annee_mois                  0.0
temp_moy                    0.0
temp_max                    0.0
temp_min                    0.0
amplitude_thermique_moy     0.0
nb_jours_gelee              0.0
nb_jours_sans_degel         0.0
nb_jours_chaud25            0.0
nb_jours_chaud30            0.0
humidite_moy                3.1
vent_moy                    3.1
vent_rafale_max             3.1
nb_jours_vent_fort          3.1
precip_total                0.0
nb_jours_pluie1mm           0.0
nb_jours_pluie5mm           0.0
ensoleillement_total        4.5
rayonnement_total           4.6
nb_jours_orage             17.4
nb_jours_brouillard         5.1
pression_moy                6.2
etp_total                   0.0
dtype: float64
